# Spark Exercises 02 — Lazy Execution & the Catalyst Plan

Polars taught you *lazy* evaluation with `scan` / `collect`. Spark takes the same idea much further: **every** DataFrame is lazy, and behind the scenes the **Catalyst optimizer** rewrites your plan before a single row is read. In this chapter you will *see* the plan with `explain()`, watch Catalyst merge filters and prune columns, and learn when to `cache()`.

**Data file:** `data/orders.csv`

### 1. **Setup** — open a local `SparkSession` (already written for you). In Foundry the session is created for you; locally we open one ourselves. `F` is the functions module — almost every column expression lives there.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("spark-course")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

### 2. Read `data/orders.csv` (`header=True`, `inferSchema=True`) into `orders`.

### 3. Build a **chain of transformations** without any action and store it in `pipeline`: keep only positive `quantity`, select a few columns, and add `revenue = quantity * unit_price`. Then evaluate `pipeline` — it returns instantly and shows only the schema. **No job ran yet.**

### 4. Now call an **action** — `pipeline.show(5)` — and the whole chain finally runs.

### 5. Print the **physical plan** with `pipeline.explain()`. Read it bottom-up: the bottom is the file scan, the top is the final step.

### 6. Print the **full set of plans** (parsed → analyzed → optimized → physical) with `pipeline.explain(mode="extended")`.

### 7. **Catalyst merges filters.** Build `two = orders.filter(quantity > 0).filter(unit_price > 5)` and call `two.explain()`. Notice the physical plan has **one** combined `Filter`, not two.

### 8. **Column pruning (projection pushdown).** Call `orders.select("order_id", "quantity").explain()` and look at the scan's `ReadSchema` — Spark only reads the two columns it needs.

### 9. How many **partitions** does `orders` have? (`orders.rdd.getNumPartitions()`) — this is how many chunks Spark processes in parallel.

### 10. Without caching, **every action recomputes the whole chain**. `cache()` the pipeline, then run an action (`count()`) to materialize it.

### 11. Check `pipeline.storageLevel` — now that it is cached you will see it is stored in memory.

### 12. Free the cache with `pipeline.unpersist()` and confirm `storageLevel` is back to none.

### 13. You can also write expressions as SQL strings. Use `selectExpr` to compute `quantity * unit_price as revenue` for 5 rows.